# Lesson 3.1 - Supervised Learning

## Objectives
- Compare supervised models for classification and regression.
- Establish baseline-first workflow.
- Contrast linear, tree, bagging, boosting, and XGBoost (if available).
## How to run (recommended)
- Restart kernel -> Run all.
- Keep your train/test split fixed while comparing models (otherwise comparisons are noisy).

## Verify
- Record one baseline metric (and the split/seed used).
- Compare two model families and explain the tradeoff (performance vs interpretability vs cost).


In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    RandomForestRegressor,
    GradientBoostingRegressor,
)

try:
    from xgboost import XGBClassifier, XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    XGBClassifier = None
    XGBRegressor = None

print("xgboost_available:", HAS_XGB)
np.random.seed(42)

xgboost_available: True


## Section A - Classification (Breast Cancer)

In [2]:
cancer = load_breast_cancer(as_frame=True)
Xc, yc = cancer.data, cancer.target

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.25, random_state=42, stratify=yc
)

models_clf = {
    "log_reg": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=3000))]),
    "knn": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=7))]),
    "decision_tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=250, random_state=42),
    "grad_boost": GradientBoostingClassifier(random_state=42),
}

if HAS_XGB:
    models_clf["xgboost"] = XGBClassifier(
        n_estimators=250,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42,
    )

rows = []
for name, model in models_clf.items():
    model.fit(Xc_train, yc_train)
    pred = model.predict(Xc_test)
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(Xc_test)[:, 1]
    else:
        scores = model.decision_function(Xc_test)
        proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
    rows.append(
        {
            "model": name,
            "accuracy": accuracy_score(yc_test, pred),
            "f1": f1_score(yc_test, pred),
            "roc_auc": roc_auc_score(yc_test, proba),
        }
    )

clf_results = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
display(clf_results)

,model,accuracy,f1,roc_auc
0,log_reg,0.986014,0.988889,0.997694
5,xgboost,0.972028,0.978022,0.996646
3,random_forest,0.958042,0.967033,0.994759
4,grad_boost,0.958042,0.967391,0.992453
1,knn,0.979021,0.983607,0.992348
2,decision_tree,0.937063,0.949721,0.918553


## Section B - Regression (Diabetes)

In [3]:
diabetes = load_diabetes(as_frame=True)
Xr, yr = diabetes.data, diabetes.target

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.25, random_state=42
)

models_reg = {
    "linear_reg": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "random_forest": RandomForestRegressor(n_estimators=250, random_state=42),
    "grad_boost": GradientBoostingRegressor(random_state=42),
}

if HAS_XGB:
    models_reg["xgboost"] = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
    )

rows = []
for name, model in models_reg.items():
    model.fit(Xr_train, yr_train)
    pred = model.predict(Xr_test)
    rows.append(
        {
            "model": name,
            "mae": mean_absolute_error(yr_test, pred),
            "rmse": mean_squared_error(yr_test, pred) ** 0.5,
            "r2": r2_score(yr_test, pred),
        }
    )

reg_results = pd.DataFrame(rows).sort_values("rmse")
display(reg_results)

,model,mae,rmse,r2
0,linear_reg,41.548507,53.369567,0.484906
1,random_forest,43.438342,54.419920,0.464432
2,grad_boost,44.260007,56.434296,0.424049
3,xgboost,45.818920,57.263024,0.407009


## Section C - Model Family Tradeoff Notes

- Linear models: strong baselines and interpretability.
- Bagging (Random Forest): robust variance reduction.
- Boosting (Gradient Boosting/XGBoost): strong tabular performance with tuning overhead.

## Business Case Studies & Exceptions
- Credit scoring may prioritize explainability over marginal AUC gain.
- House-price regression often benefits from nonlinear ensembles due to interaction effects.

## Interview Questions & Answers
1. Bagging vs boosting? Bagging averages parallel learners; boosting builds learners sequentially to reduce residual errors.
2. When prefer linear models? Governance, low-latency requirements, or when nonlinear lift is limited.